## 4. Client Test

**NOTE**: 

The client test resides in the `client` directory and mounted to the devcontainer.
That means you can either run this test either server-side or client-side.
There are 3 different ways to run this test:

a) **If you are running from gai-chat-svr devcontainer**:  

    - Press **F5** to start the chat server.    
    - Set base url to **http://localhost:12031** in client_config  
    NOTE: You cannot run the server using `docker-compose up` command in this case, as it will conflict with the devcontainer's port mapping.

b) **If you are running from gai-chat-svr Client**:  

    - Use `docker-compose up` to start the chat server.  
    - Set base url to **http://localhost:12031** in client_config  

c) **If you are running from gai-sdk devcontainer**:  

    - From gai-sdk, open devcontainer.  
    - Set base url to **http://gai-chat-svr:12031** in client_config  

### b) Load the client

In [1]:
import os
os.environ["LOG_LEVEL"] = "INFO"
from gai.chat.client import ChatClient
chat_client = ChatClient({
        "client_type": "gai",
        #"url": "http://localhost:12031/gen/v1/chat/completions"
        "url": "http://gai-chat-svr:12031/gen/v1/chat/completions"
        })


### c) Stream

In [2]:
result = chat_client.chat(model="ttt",messages="User:Tell me a one paragraph story",stream=True)
for chunk in result:
    print(chunk.extract(),end="",flush=True)

Once upon a time, in a small village nestled between two vast mountains, there lived a poor woodcutter named John. John worked every day from dawn till dusk, hacking away at the trees with his trusty axe, hoping to earn enough coins to feed his family. Despite his tireless efforts, John was often left with barely enough for a warm meal and a bed of straw. His children, Emily and young Thomas, would often listen to their mother's stories of better days, their eyes wide with wonder and hope. One fateful winter, as the snow piled high and the winds howled, John stumbled upon something extraordinary while sawing through a frozen tree: a golden pocket watch, its face gleaming with intricate engravings. The watch was so unlike anything he had ever seen before, and its sudden appearance in the cold wilderness left John both bewildered and awestruck. As he held the watch in his weathered hands, the once-poor woodcutter realized that fate had given him a new chance at a brighter future.{'type':

### d) Generate

In [3]:
result = chat_client.chat(model="ttt",messages="User:Tell me a one paragraph story",stream=False)
print(result.extract()["content"])

Once upon a time, in a small sleepy town, there lived a girl named Lily. She was nine years old, with long brown hair and bright blue eyes. Lily loved to play in the woods behind her house, exploring the hidden trails and secret streams. One day, while wandering deeper into the woods than she had ever been before, she stumbled upon an old, mysterious-looking door hidden behind a thick veil of ivy. Curiosity piqued, Lily pushed aside the tangled vines and discovered that the door was unlocked. As she pushed it open, a warm golden light spilled out, and she stepped into a magical world filled with talking animals, enchanted forests, and brave knights. In this new world, Lily learned valuable lessons about courage, friendship, and the power of believing in oneself. She returned home only after a year, with a heart full of unforgettable memories and an unquenchable thirst for adventure.


### e) Tool Call

In [4]:
result = chat_client.chat(
        model="ttt",
        messages=[
            {"role":"user","content":"What is the current time in Singapore?"},
            {"role":"assistant","content":""}
        ],
        tool_choice="required",
        tools=[
            {
                "type": "function",
                "function": {
                    "name": "google",
                    "description": "The 'google' function is a powerful tool that allows the AI to gather external information from the internet using Google search. It can be invoked when the AI needs to answer a question or provide information that requires up-to-date, comprehensive, and diverse sources which are not inherently known by the AI. For instance, it can be used to find current date, current news, weather updates, latest sports scores, trending topics, specific facts, or even the current date and time. The usage of this tool should be considered when the user's query implies or explicitly requests recent or wide-ranging data, or when the AI's inherent knowledge base may not have the required or most current information. The 'search_query' parameter should be a concise and accurate representation of the information needed.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "search_query": {
                                "type": "string",
                                "description": "The search query to search google with. For example, to find the current date or time, use 'current date' or 'current time' respectively."
                            }
                        },
                        "required": ["search_query"]
                    }
                }
            }
        ],
        stream=False)
print(result.extract())

{'type': 'function', 'name': 'google', 'arguments': '{"search_query": "current time in Singapore"}'}


### f) Structured Output

In [5]:
from pydantic import BaseModel
class Book(BaseModel):
    title: str
    summary: str
    author: str
    published_year: int

text = """Foundation is a science fiction novel by American writer
Isaac Asimov. It is the first published in his Foundation Trilogy (later
expanded into the Foundation series). Foundation is a cycle of five
interrelated short stories, first published as a single book by Gnome Press
in 1951. Collectively they tell the early story of the Foundation,
an institute founded by psychohistorian Hari Seldon to preserve the best
of galactic civilization after the collapse of the Galactic Empire.
"""

result = chat_client.chat(
    model="ttt",
    messages=[{'role':'user','content':text},{'role':'assistant','content':''}], 
    json_schema=Book.schema(),
    stream=False)
print(result.extract()["content"])

/tmp/ipykernel_96508/2285534030.py:20: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  json_schema=Book.schema(),


{
  "title": "Foundation",
  "summary": "Foundation is a science fiction novel by American writer Isaac Asimov. It is the first published in his Foundation Trilogy (later expanded into the Foundation series). Foundation is a cycle of five interrelated short stories, first published as a single book by Gnome Press in 1951. Collectively they tell the early story of the Foundation, an institute founded by psychohistorian Hari Seldon to preserve the best of galactic civilization after the collapse of the Galactic Empire.",
  "author": "Isaac Asimov",
  "published_year": 1951
}
